In [51]:
import numpy as np 
import pandas as pd 
import matplotlib.pyplot as plt 
import seaborn as sns 
import warnings

warnings.filterwarnings('ignore')

In [52]:
df = pd.read_csv("../data/Cleaned/cleaned_dataco.csv")

In [53]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 180519 entries, 0 to 180518
Data columns (total 45 columns):
 #   Column                         Non-Null Count   Dtype  
---  ------                         --------------   -----  
 0   Type                           180519 non-null  object 
 1   Days for shipping (real)       180519 non-null  int64  
 2   Days for shipment (scheduled)  180519 non-null  int64  
 3   Benefit per order              180519 non-null  float64
 4   Sales per customer             180519 non-null  float64
 5   Delivery Status                180519 non-null  object 
 6   Late_delivery_risk             180519 non-null  int64  
 7   Category Id                    180519 non-null  int64  
 8   Category Name                  180519 non-null  object 
 9   Customer City                  180519 non-null  object 
 10  Customer Country               180519 non-null  object 
 11  Customer Id                    180519 non-null  int64  
 12  Customer Segment              

In [54]:
df['order date (DateOrders)'] = pd.to_datetime(df['order date (DateOrders)'])
df['shipping date (DateOrders)'] = pd.to_datetime(df['shipping date (DateOrders)'])

How many days early or late was an order delivered?

In [55]:
df['Delivery Delay'] = (df['Days for shipping (real)'] - df['Days for shipment (scheduled)'])

In [56]:
df[['Days for shipping (real)',
    'Days for shipment (scheduled)',
    'Delivery Delay']].head()

,Days for shipping (real),Days for shipment (scheduled),Delivery Delay
0,3,4,-1
1,5,4,1
2,4,4,0
3,3,4,-1
4,2,4,-2


Which orders are actually profitable?

In [57]:
df['Profit Margin (%)'] = np.where(
    df['Sales'] != 0,
    (df['Order Profit Per Order'] / df['Sales']) * 100,
    0
).round(2)

In [58]:
df['Profit Margin (%)'].describe()

count    180519.000000
mean         10.832485
std          42.059324
min        -275.000000
25%           6.220000
50%          24.250000
75%          33.600000
max          50.040000
Name: Profit Margin (%), dtype: float64

Order Year

In [59]:
df['Order Year'] = df['order date (DateOrders)'].dt.year

In [60]:
df.head()

,Type,Days for shipping (real),Days for shipment (scheduled),Benefit per order,Sales per customer,Delivery Status,Late_delivery_risk,Category Id,Category Name,Customer City,...,Product Card Id,Product Category Id,Product Name,Product Price,Product Status,shipping date (DateOrders),Shipping Mode,Delivery Delay,Profit Margin (%),Order Year
0,DEBIT,3,4,91.250000,314.640015,Advance shipping,0,73,Sporting Goods,Caguas,...,1360,73,Smart watch,327.75,0,2018-02-03 22:56:00,Standard Class,-1,27.84,2018
1,TRANSFER,5,4,-249.089996,311.359985,Late delivery,1,73,Sporting Goods,Caguas,...,1360,73,Smart watch,327.75,0,2018-01-18 12:27:00,Standard Class,1,-76.00,2018
2,CASH,4,4,-247.779999,309.720001,Shipping on time,0,73,Sporting Goods,San Jose,...,1360,73,Smart watch,327.75,0,2018-01-17 12:06:00,Standard Class,0,-75.60,2018
3,DEBIT,3,4,22.860001,304.809998,Advance shipping,0,73,Sporting Goods,Los Angeles,...,1360,73,Smart watch,327.75,0,2018-01-16 11:45:00,Standard Class,-1,6.97,2018
4,PAYMENT,2,4,134.210007,298.250000,Advance shipping,0,73,Sporting Goods,Caguas,...,1360,73,Smart watch,327.75,0,2018-01-15 11:24:00,Standard Class,-2,40.95,2018


Order Month

In [61]:
df['Order Month']= df['order date (DateOrders)'].dt.month_name()

In [62]:
df['Order Month'].value_counts()

Order Month
January      17979
May          15976
July         15922
March        15919
August       15912
September    15489
April        15435
June         15139
February     14529
October      12955
December     12764
November     12500
Name: count, dtype: int64

In [63]:
df['Order Quarter'] = df['order date (DateOrders)'].dt.quarter

In [64]:
df['Weekday'] = df['order date (DateOrders)'].dt.day_name()

In [65]:
df['Weekend Order'] = np.where(
    df['Weekday'].isin(['Saturday', 'Sunday']),
    'Yes', 
    'No'
)

High Value Order

In [66]:
threshold = df['Sales'].quantile(0.75)

df['High Value Order']= np.where(
    df['Sales']>=threshold,
    'Yes', 
    'No'
)

In [67]:
df['High Value Order'].value_counts()

High Value Order
No     133179
Yes     47340
Name: count, dtype: int64

Profit Category

In [68]:
conditions =[
    df['Order Profit Per Order']<0,
    df['Order Profit Per Order']==0, 
    df['Order Profit Per Order']>0, 
]
choices=[
    'Loss', 
    'Break-even', 
    'Profit'
]

df['Profit Category']= np.select(
    conditions,
    choices, 
    default='Unknown'
)

In [69]:
df['Profit Category'].value_counts()

Profit Category
Profit        145558
Loss           33784
Break-even      1177
Name: count, dtype: int64

In [70]:
df['Delivery Performance'] = np.select(
    [
        df['Delivery Delay'] < 0,
        df['Delivery Delay'] == 0,
        df['Delivery Delay'] > 0
    ],
    [
        'Early',
        'On Time',
        'Late'
    ],
    default='Unknown'
)

Order Size

In [71]:
q1 = df['Sales'].quantile(0.25)
q3 = df['Sales'].quantile(0.75)

conditions = [
    df['Sales']<= q1,
    (df['Sales']>q1) & (df['Sales']<=q3), 
    df['Sales']>q3
]

choices= [
    'Small',
    'Medium', 
    'Large'
]

df['Order Size']= np.select(
    conditions,
    choices,
    default='Unknown'
)

Discount Category

In [72]:
df['Discount Category'] = pd.cut(
    df['Order Item Discount Rate'],
    bins=[-0.01,0,0.1,0.2,1], 
    labels=[
        'No Discount',
        'Low',
        'Medium',
        'High'
    ]
)

Shipping Efficiency


In [73]:
df['Shipping Efficiency'] = (
    df['Days for shipment (scheduled)'] / df['Days for shipping (real)']
).round(2)

Profitability Segment

In [74]:
df['Profit Segment']=pd.qcut(
    df['Order Profit Per Order'], 
    q=4, 
    labels=[
        'Low',
        'Medium',
        'High',
        'Very High'
    ]
)

Customer Lifetime Revenue

In [75]:
customer_revenue = (
    df.groupby('Customer Id')['Sales']
      .transform('sum')
)

df['Customer Lifetime Revenue'] = customer_revenue

Customer Order Count

In [76]:
customer_orders = (
    df.groupby('Customer Id')['Order Id']
      .transform('count')
)

df['Customer Order Count'] = customer_orders

Average Order Value per Customer

In [77]:
df['Customer Average Order Value'] =(
    df['Customer Lifetime Revenue'] / df['Customer Order Count']
)

Revenue Contribution

In [78]:
total_sales = df['Sales'].sum()

df['Revenue Contribution (%)'] = (
    df['Sales']
    /
    total_sales
) * 100

Order Age

In [79]:
df['Order Age']= (
    df['shipping date (DateOrders)'] - df['order date (DateOrders)']
).dt.days

In [80]:
new_features = [
    'Delivery Delay',
    'Profit Margin (%)',
    'Order Year',
    'Order Month',
    'Order Quarter',
    'Weekday',
    'Weekend Order',
    'High Value Order',
    'Profit Category'
]

df[new_features].head()

,Delivery Delay,Profit Margin (%),Order Year,Order Month,Order Quarter,Weekday,Weekend Order,High Value Order,Profit Category
0,-1,27.84,2018,January,1,Wednesday,No,Yes,Profit
1,1,-76.00,2018,January,1,Saturday,Yes,Yes,Loss
2,0,-75.60,2018,January,1,Saturday,Yes,Yes,Loss
3,-1,6.97,2018,January,1,Saturday,Yes,Yes,Profit
4,-2,40.95,2018,January,1,Saturday,Yes,Yes,Profit


In [81]:
new_features = [
    'Delivery Delay',
    'Profit Margin (%)',
    'Order Year',
    'Order Month',
    'Order Quarter',
    'Weekday',
    'Weekend Order',
    'High Value Order',
    'Profit Category',
    'Delivery Performance', 
    'Order Size',
    'Discount Category', 
    'Shipping Efficiency', 
    'Profit Segment', 
    'Customer Lifetime Revenue', 
    'Customer Order Count', 
    'Customer Average Order Value', 
    'Revenue Contribution (%)', 
    'Order Age'
]

df[new_features].head()

,Delivery Delay,Profit Margin (%),Order Year,Order Month,Order Quarter,Weekday,Weekend Order,High Value Order,Profit Category,Delivery Performance,Order Size,Discount Category,Shipping Efficiency,Profit Segment,Customer Lifetime Revenue,Customer Order Count,Customer Average Order Value,Revenue Contribution (%),Order Age
0,-1,27.84,2018,January,1,Wednesday,No,Yes,Profit,Early,Large,Low,1.33,Very High,327.75,1,327.75,0.000891,3
1,1,-76.00,2018,January,1,Saturday,Yes,Yes,Loss,Late,Large,Low,0.80,Low,327.75,1,327.75,0.000891,5
2,0,-75.60,2018,January,1,Saturday,Yes,Yes,Loss,On Time,Large,Low,1.00,Low,327.75,1,327.75,0.000891,4
3,-1,6.97,2018,January,1,Saturday,Yes,Yes,Profit,Early,Large,Low,1.33,Medium,327.75,1,327.75,0.000891,3
4,-2,40.95,2018,January,1,Saturday,Yes,Yes,Profit,Early,Large,Low,2.00,Very High,327.75,1,327.75,0.000891,2


In [82]:
df.to_csv('../data/Cleaned/feature_engineered_dataco.csv', index= False)

In [83]:
df = df.drop(columns=['Weekday',
'Weekend Order',
'High Value Order',
'Profit Category',
'Revenue Contribution (%)'])

In [84]:
df.head()

,Type,Days for shipping (real),Days for shipment (scheduled),Benefit per order,Sales per customer,Delivery Status,Late_delivery_risk,Category Id,Category Name,Customer City,...,Order Quarter,Delivery Performance,Order Size,Discount Category,Shipping Efficiency,Profit Segment,Customer Lifetime Revenue,Customer Order Count,Customer Average Order Value,Order Age
0,DEBIT,3,4,91.250000,314.640015,Advance shipping,0,73,Sporting Goods,Caguas,...,1,Early,Large,Low,1.33,Very High,327.75,1,327.75,3
1,TRANSFER,5,4,-249.089996,311.359985,Late delivery,1,73,Sporting Goods,Caguas,...,1,Late,Large,Low,0.80,Low,327.75,1,327.75,5
2,CASH,4,4,-247.779999,309.720001,Shipping on time,0,73,Sporting Goods,San Jose,...,1,On Time,Large,Low,1.00,Low,327.75,1,327.75,4
3,DEBIT,3,4,22.860001,304.809998,Advance shipping,0,73,Sporting Goods,Los Angeles,...,1,Early,Large,Low,1.33,Medium,327.75,1,327.75,3
4,PAYMENT,2,4,134.210007,298.250000,Advance shipping,0,73,Sporting Goods,Caguas,...,1,Early,Large,Low,2.00,Very High,327.75,1,327.75,2


In [85]:
df.columns.to_list()

['Type',
 'Days for shipping (real)',
 'Days for shipment (scheduled)',
 'Benefit per order',
 'Sales per customer',
 'Delivery Status',
 'Late_delivery_risk',
 'Category Id',
 'Category Name',
 'Customer City',
 'Customer Country',
 'Customer Id',
 'Customer Segment',
 'Customer State',
 'Customer Zipcode',
 'Department Id',
 'Department Name',
 'Latitude',
 'Longitude',
 'Market',
 'Order City',
 'Order Country',
 'Order Customer Id',
 'order date (DateOrders)',
 'Order Id',
 'Order Item Cardprod Id',
 'Order Item Discount',
 'Order Item Discount Rate',
 'Order Item Id',
 'Order Item Product Price',
 'Order Item Profit Ratio',
 'Order Item Quantity',
 'Sales',
 'Order Item Total',
 'Order Profit Per Order',
 'Order Region',
 'Order State',
 'Order Status',
 'Product Card Id',
 'Product Category Id',
 'Product Name',
 'Product Price',
 'Product Status',
 'shipping date (DateOrders)',
 'Shipping Mode',
 'Delivery Delay',
 'Profit Margin (%)',
 'Order Year',
 'Order Month',
 'Order Quar

In [86]:
(df['Customer Id'] == df['Order Customer Id']).all()

np.True_

In [87]:
df = df.drop(columns=['Order Customer Id'])

In [88]:
df.isnull().sum()

Type                                0
Days for shipping (real)            0
Days for shipment (scheduled)       0
Benefit per order                   0
Sales per customer                  0
Delivery Status                     0
Late_delivery_risk                  0
Category Id                         0
Category Name                       0
Customer City                       0
Customer Country                    0
Customer Id                         0
Customer Segment                    0
Customer State                      0
Customer Zipcode                    0
Department Id                       0
Department Name                     0
Latitude                            0
Longitude                           0
Market                              0
Order City                          0
Order Country                       0
order date (DateOrders)             0
Order Id                            0
Order Item Cardprod Id              0
Order Item Discount                 0
Order Item D

In [89]:
print(df['Days for shipping (real)'].isna().sum())
print(df['Days for shipment (scheduled)'].isna().sum())
print((df['Days for shipping (real)'] == 0).sum())

0
0
5080


In [90]:
df = df.drop(columns=['Shipping Efficiency'])

In [91]:
df = df.drop(columns=['Sales per customer'])

In [92]:
df = df.drop(columns=['Benefit per order'])

In [93]:
df['Profit Margin (%)'].describe()

count    180519.000000
mean         10.832485
std          42.059324
min        -275.000000
25%           6.220000
50%          24.250000
75%          33.600000
max          50.040000
Name: Profit Margin (%), dtype: float64

In [94]:
df[df['Profit Margin (%)'] < 0][['Sales', 'Order Profit Per Order', 'Profit Margin (%)']].head()

,Sales,Order Profit Per Order,Profit Margin (%)
1,327.75,-249.089996,-76.00
2,327.75,-247.779999,-75.60
15,327.75,-259.579987,-79.20
16,327.75,-246.360001,-75.17
28,327.75,-17.139999,-5.23


In [95]:
df.columns.tolist()

['Type',
 'Days for shipping (real)',
 'Days for shipment (scheduled)',
 'Delivery Status',
 'Late_delivery_risk',
 'Category Id',
 'Category Name',
 'Customer City',
 'Customer Country',
 'Customer Id',
 'Customer Segment',
 'Customer State',
 'Customer Zipcode',
 'Department Id',
 'Department Name',
 'Latitude',
 'Longitude',
 'Market',
 'Order City',
 'Order Country',
 'order date (DateOrders)',
 'Order Id',
 'Order Item Cardprod Id',
 'Order Item Discount',
 'Order Item Discount Rate',
 'Order Item Id',
 'Order Item Product Price',
 'Order Item Profit Ratio',
 'Order Item Quantity',
 'Sales',
 'Order Item Total',
 'Order Profit Per Order',
 'Order Region',
 'Order State',
 'Order Status',
 'Product Card Id',
 'Product Category Id',
 'Product Name',
 'Product Price',
 'Product Status',
 'shipping date (DateOrders)',
 'Shipping Mode',
 'Delivery Delay',
 'Profit Margin (%)',
 'Order Year',
 'Order Month',
 'Order Quarter',
 'Delivery Performance',
 'Order Size',
 'Discount Category',

In [96]:
df.dtypes

Type                                     object
Days for shipping (real)                  int64
Days for shipment (scheduled)             int64
Delivery Status                          object
Late_delivery_risk                        int64
Category Id                               int64
Category Name                            object
Customer City                            object
Customer Country                         object
Customer Id                               int64
Customer Segment                         object
Customer State                           object
Customer Zipcode                        float64
Department Id                             int64
Department Name                          object
Latitude                                float64
Longitude                               float64
Market                                   object
Order City                               object
Order Country                            object
order date (DateOrders)          datetim